In [1]:
# =========================================================
# STEP 1: LOAD DATA
# =========================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

DATA_PATH = r"../data/processed/emi_featured_final.csv"
df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)

# Use sample for speed
df = df.sample(50000, random_state=42)

print("Sample shape:", df.shape)

Original shape: (404800, 49)
Sample shape: (50000, 49)


In [8]:
# =========================================================
# STEP 2: DEFINE FEATURES AND TARGET
# =========================================================

drop_cols = [
    "max_monthly_emi",
    "emi_capacity_gap",
    "requested_emi_estimate",
    "requested_emi_to_income_ratio"
]
X = df.drop(columns=drop_cols)
y = df["max_monthly_emi"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (50000, 45)
y shape: (50000,)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (40000, 45)
Test : (10000, 45)


In [10]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical:", len(categorical_cols))
print("Numerical:", len(numerical_cols))

Categorical: 12
Numerical: 33


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=80,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=80,
        learning_rate=0.1,
        random_state=42
    ),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=10,
        random_state=42
    )
}

In [13]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

results = []
trained_models = {}

for name, model in models.items():
    print(f"\nTraining: {name}")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "MAPE (%)": mape
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values(by="R2", ascending=False)

for col in ["RMSE", "MAE", "R2", "MAPE (%)"]:
    results_df[col] = results_df[col].round(4)

print("\nRegression Results:")
print(results_df)


Training: Linear Regression

Training: Random Forest

Training: Gradient Boosting

Training: Decision Tree

Regression Results:
               Model       RMSE        MAE      R2  MAPE (%)
1      Random Forest   950.7997   376.6882  0.9853    7.0806
2  Gradient Boosting  1090.4956   535.6075  0.9807   22.9104
3      Decision Tree  1338.4652   579.9197  0.9709   10.2307
0  Linear Regression  3290.9839  2261.5357  0.8242  128.7544


Four regression models were trained to predict maximum monthly EMI: Linear Regression, Random Forest Regressor, Gradient Boosting Regressor, and Decision Tree Regressor.

After removing data leakage features, Random Forest Regressor achieved the best performance with the lowest RMSE (950.8), lowest MAE (376.7), highest R² (0.9853), and lowest MAPE (7.08%).

Gradient Boosting and Decision Tree also performed well, but with higher prediction errors. Linear Regression showed significantly lower performance, indicating that the relationship between financial variables and EMI amount is non-linear.

Therefore, Random Forest Regressor was selected as the final model for predicting maximum monthly EMI.

Care was taken to remove features derived from the target variable to prevent data leakage and ensure realistic model performance.

In [14]:
# =========================================================
# STEP 8: MLFLOW LOGGING - REGRESSION
# =========================================================

import mlflow
import mlflow.sklearn

mlflow.set_experiment("EMI_Predict_AI_Regression")

regression_results_df = results_df.copy()

for _, row in regression_results_df.iterrows():
    model_name = row["Model"]
    model = trained_models[model_name]

    with mlflow.start_run(run_name=f"reg_{model_name}"):
        mlflow.log_param("problem_type", "regression")
        mlflow.log_param("model_type", model_name)

        mlflow.log_metric("rmse", float(row["RMSE"]))
        mlflow.log_metric("mae", float(row["MAE"]))
        mlflow.log_metric("r2", float(row["R2"]))
        mlflow.log_metric("mape", float(row["MAPE (%)"]))

        mlflow.sklearn.log_model(model, artifact_path="model")

print("Regression models logged to MLflow successfully.")

2026/04/08 19:08:37 INFO mlflow.tracking.fluent: Experiment with name 'EMI_Predict_AI_Regression' does not exist. Creating a new experiment.
2026/04/08 19:08:37 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:


Regression models logged to MLflow successfully.


Final Conclusion:
Among the regression models tested, Random Forest Regressor achieved the best performance after removing target-derived leakage features. It was selected as the final model for maximum EMI prediction because it delivered the lowest RMSE and MAE and the highest R² score.

MLflow was used to track classification and regression experiments separately. It logged model metrics, parameters, and model artifacts, enabling systematic comparison and final model selection in an organized and reproducible way.

In [15]:
# =========================================================
# SAVE FINAL REGRESSION MODEL
# =========================================================

import os
import joblib

os.makedirs(r"models", exist_ok=True)

best_regression_model = trained_models["Random Forest"]

joblib.dump(
    best_regression_model,
    r"models/best_regression_model.pkl"
)

print("Best regression model saved successfully.")

Best regression model saved successfully.
